# Compare three ways of computing marginal Matern Kernel Matrix

In [1]:
%run -i ~/project/preambles
%run -i ~/project/helper_functions
%run -i ~/project/fitting_functions

cuda


In [2]:
# Define a threshold for the relative squared Frobenius norm
threshold = 1e-3

# Generate a simple test case (e.g., 5 points in 2D space) and move it to the device
X = torch.randn(5, 2, device=device, dtype=torch.float64)

# Define test parameters for the Matérn kernel as tensors and move them to the device
nu_i = torch.tensor(1.5, dtype=torch.float64, device=device)
alpha_i = torch.tensor(1.0, dtype=torch.float64, device=device)
sigma_i = torch.tensor(10.0, dtype=torch.float64, device=device)


# Compute the ground truth matrix using matern_kernel
K_true = matern_kernel(torch.cdist(X, X), nu_i, alpha_i, sigma_i)

# Compute the approximate matrix using approx_matern_kernel_marginal
K_approx = approx_matern_kernel_marginal(X, nu_i, alpha_i, sigma_i)

# Compute the approximate matrix using approx_matern_kernel_marginal_old
K_approx_old = approx_matern_kernel_marginal_old(X, nu_i, alpha_i, sigma_i)

# Function to compute relative squared Frobenius norm
def relative_frobenius_norm(K_approx, K_true):
    return torch.norm(K_approx - K_true, 'fro')**2 / torch.norm(K_true, 'fro')**2

# Compare the two approximate methods to the ground truth
relative_norm_approx = relative_frobenius_norm(K_approx, K_true)
relative_norm_approx_old = relative_frobenius_norm(K_approx_old, K_true)

# Output success or failure based on the threshold
if relative_norm_approx > threshold:
    print(f"Failure: approx_matern_kernel_marginal exceeds the threshold. Relative norm: {relative_norm_approx}")
else:
    print(f"Success: approx_matern_kernel_marginal is within the threshold. Relative norm: {relative_norm_approx}")

if relative_norm_approx_old > threshold:
    print(f"Failure: approx_matern_kernel_marginal_old exceeds the threshold. Relative norm: {relative_norm_approx_old}")
else:
    print(f"Success: approx_matern_kernel_marginal_old is within the threshold. Relative norm: {relative_norm_approx_old}")

Failure: approx_matern_kernel_marginal exceeds the threshold. Relative norm: 0.02152396484828849
Success: approx_matern_kernel_marginal_old is within the threshold. Relative norm: 1.5707911311600398e-07


In [3]:
K_approx_old

tensor([[100.0000,  78.0337,  38.3333,  69.0305,  79.5741],
        [ 78.0337, 100.0000,  59.7204,  75.7953,  98.3502],
        [ 38.3333,  59.7204, 100.0000,  44.7650,  58.7194],
        [ 69.0305,  75.7953,  44.7650, 100.0000,  71.6303],
        [ 79.5741,  98.3502,  58.7194,  71.6303, 100.0000]], device='cuda:0',
       dtype=torch.float64)

In [4]:
K_approx

tensor([[100.0000,  69.6233,  20.6792,  56.9228,  71.8484],
        [ 69.6233, 100.0000,  44.5893,  66.4116,  98.2859],
        [ 20.6792,  44.5893, 100.0000,  27.1239,  43.3235],
        [ 56.9228,  66.4116,  27.1239, 100.0000,  60.5254],
        [ 71.8484,  98.2859,  43.3235,  60.5254, 100.0000]], device='cuda:0',
       dtype=torch.float64)

In [5]:
K_true

tensor([[100.0000,  78.0603,  38.3333,  69.0321,  79.5859],
        [ 78.0603, 100.0000,  59.6735,  75.8568,  98.3502],
        [ 38.3333,  59.6735, 100.0000,  44.7882,  58.6630],
        [ 69.0321,  75.8568,  44.7882, 100.0000,  71.5996],
        [ 79.5859,  98.3502,  58.6630,  71.5996, 100.0000]], device='cuda:0',
       dtype=torch.float64)